# 19-тарау. Мектептің RAG-көмекшісі

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 19.1. Жобаның мақсаты

## 19.2. 1-қадам. Құжаттар

In [ ]:
import json
import random
import re
import urllib.request
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
urllib.request.urlretrieve(RAW + "data/mektep.json", "mektep.json")
docs = json.load(open("mektep.json", encoding="utf-8"))
sections = [dict(s, doc=d["doc"]) for d in docs for s in d["sections"]]
print(len(docs), "құжат,", len(sections), "бөлім\n")
for d in docs:
    print(f"{d['doc']}: {', '.join(s['title'] for s in d['sections'])}")
print()
print(json.dumps(sections[8], ensure_ascii=False, indent=1))

## 19.3. 2-қадам. Индекс және сынақ сұрақтары

In [ ]:
def make_text(s, mode):
    if mode == "мәтін":
        return s["text"]
    text = f"{s['doc']}. {s['title']}. {s['text']}"
    return text if mode == "+ атаулар" else text + " " + " ".join(s["faq"])

class Index:
    def __init__(self, mode):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
        self.M = self.vec.fit_transform([make_text(s, mode) for s in sections])

    def search(self, question, k=3):
        scores = (self.M @ self.vec.transform([question]).T).toarray().ravel()
        best = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in best]

In [ ]:
QA = [
    ("Сабақ нешеде басталады?", ["Сабақ уақыты", "Сабақтардың уақыты"]),
    ("Үлкен үзіліс қанша минутқа созылады?", ["Үзілістер"]),
    ("Бесінші сабақ қашан бітеді?", ["Сабақтардың уақыты"]),
    ("Сабақта телефон қолдануға бола ма?", ["Ұялы телефон"]),
    ("Кешігіп қалсам не істеуім керек?", ["Кешігу"]),
    ("Ауырып қалсам, мектепке кім хабарлайды?", ["Сабақты босату"]),
    ("Мектеп формасы қандай түсті?", ["Мектеп формасы"]),
    ("Кітапты неше күнге береді?", ["Кітап алу"]),
    ("Кітапхана қай қабатта?", ["Жұмыс уақыты және орны"]),
    ("Оқулықтарды қашан қайтарамыз?", ["Оқулықтар"]),
    ("Кітапты жоғалтып алсам не болады?", ["Жоғалған кітап"]),
    ("Асхана қайда орналасқан?", ["Орны және уақыты"]),
    ("Бастауыш сынып балаларына тамақ тегін бе?", ["Тегін тамақ"]),
    ("Түскі ас қанша тұрады?", ["Баға және мәзір"]),
    ("Балам сүт ішпейді, аллергиясы бар. Кімге айтамыз?", ["Тағамдық аллергия"]),
    ("Роботтар жасауды қай жерде үйренуге болады?", ["Робототехника"]),
    ("Python үйрететін үйірме бар ма?", ["Жасанды интеллект және бағдарламалау"]),
    ("Үйірмеге қалай жазыламын?", ["Үйірмеге жазылу"]),
    ("Домбыра тартуды үйренгім келеді, бірақ аспабым жоқ", ["Домбыра және хор"]),
    ("Олимпиаданың мектеп кезеңі қай айда өтеді?",
     ["Пән олимпиадаларының кезеңдері"]),
    ("Облыстық олимпиада қашан?", ["Пән олимпиадаларының кезеңдері"]),
    ("ЖИ олимпиадасына қалай дайындалуға болады?",
     ["Жасанды интеллект олимпиадалары", "Жасанды интеллект және бағдарламалау"]),
    ("ТЖБ қашан жазылады?", ["Жиынтық бағалау"]),
    ("Жылдық баға қалай шығарылады?", ["Тоқсандар"]),
    ("9-сыныпта емтихан тапсырамыз ба?", ["Қорытынды аттестаттау"]),
    ("Мектепте оқитыным туралы қағаз керек. Қанша күнде дайын болады?",
     ["Анықтамалар"]),
    ("Мектептің электрондық поштасы қандай?", ["Мектеп кеңсесі"]),
    ("Курткамды мектепте ұмытып кеттім, қайдан іздеймін?", ["Жоғалған заттар"]),
    ("Психологпен сөйлескім келеді, қалай жазыламын?", ["Мектеп психологы"]),
    ("Бағамды қайдан көремін?", ["Бағаларды көру"]),
]
NOT_IN_DOCS = [
    "Ертең ауа райы қандай болады?", "Алгебрадан үй тапсырмасы қандай?",
    "Айгерімнің телефон нөмірі қандай?", "Футболдан кеше кім жеңді?",
    "ҰБТ-да неше пән бар?", "Мектепке жақын дәріхана қайда?",
    "Математика мұғалімі неше жаста?", "Жазғы лагерьге жолдама қанша тұрады?",
]

In [ ]:
def evaluate(index):
    top1 = top3 = rr = 0
    for q, gold in QA:
        titles = [sections[i]["title"] for i, _ in index.search(q, k=len(sections))]
        rank = min(titles.index(g) for g in gold) + 1
        top1 += rank == 1
        top3 += rank <= 3
        rr += 1 / rank
    return top1, top3, rr / len(QA)

scores = {}
for mode in ["мәтін", "+ атаулар", "+ сұрақтар"]:
    scores[mode] = evaluate(Index(mode))
    t1, t3, mrr = scores[mode]
    print(f"{mode:11} бірінші: {t1}/30   үштік: {t3}/30   MRR: {mrr:.3f}")

## 19.4. 3-қадам. Жиі қойылатын сұрақтар не үшін керек?

In [ ]:
index = Index("+ сұрақтар")
for q, gold in QA:
    hits = index.search(q)
    if sections[hits[0][0]]["title"] not in gold:
        found = ", ".join(f"{sections[i]['title']} ({s:.2f})" for i, s in hits[:2])
        print(f"{q}\n   күтілген: {gold[0]}\n   табылған: {found}")

## 19.5. 4-қадам. Жеке деректерді қорғау

In [ ]:
PRIVATE = r"телефон нөмір|мекенжай|неше жаста|жасы қанша|үйі қайда"

for q in ["Айгерімнің телефон нөмірі қандай?", "Сабақта телефон қолдануға бола ма?"]:
    blocked = bool(re.search(PRIVATE, q.lower()))
    print(f"{'ТОҚТАТЫЛДЫ' if blocked else 'іздеуге жіберілді'}: {q}")

## 19.6. 5-қадам. «Білмеймін» шегі

In [ ]:
def is_correct(q, gold, thr):
    if re.search(PRIVATE, q.lower()):
        return gold is None
    i, score = index.search(q, 1)[0]
    if score < thr:
        return gold is None
    return gold is not None and sections[i]["title"] in gold

items = [(q, g) for q, g in QA] + [(q, None) for q in NOT_IN_DOCS]
random.seed(1)
random.shuffle(items)
dev, test = items[:19], items[19:]

thresholds = np.round(np.arange(0.10, 0.36, 0.02), 2)
sweep = [(t, sum(is_correct(q, g, t) for q, g in dev),
          sum(is_correct(q, g, t) for q, g in test)) for t in thresholds]
THRESHOLD, best_dev, _ = max(sweep, key=lambda r: r[1])
test_ok = sum(is_correct(q, g, THRESHOLD) for q, g in test)
print(f"шек: {THRESHOLD}   dev: {best_dev}/{len(dev)}   test: {test_ok}/{len(test)}")

## 19.7. 6-қадам. Көмекші

In [ ]:
CONTACT = "Мектеп кеңсесіне хабарласыңыз: kense@bilim-mektep.example"

def assistant(question):
    if re.search(PRIVATE, question.lower()):
        return "Кешіріңіз, адамдардың жеке деректері туралы ақпарат бермеймін."
    i, score = index.search(question, 1)[0]
    if score < THRESHOLD:
        return "Мектеп құжаттарынан бұл сұраққа жауап таппадым. " + CONTACT
    s = sections[i]
    return f"{s['text']}\n   (Дереккөз: «{s['doc']}» → «{s['title']}»)"

for q in ["Кітапхана сенбіде жұмыс істей ме?", "Үйірмелер ақылы ма?",
          "Мектепке жақын дәріхана қайда?", "Математика мұғалімі неше жаста?"]:
    print("—", q, "\n  ", assistant(q))

## 19.8. 7-қадам. Тілдік модель және веб-интерфейс

**Міндетті емес ұяшық.** Бұл код қосымша бағдарламаны, интернеттегі қызметті немесе өз файлыңызды қажет етеді (толығырақ кітапта). Іске қосу үшін ұяшықтың бірінші жолын (`%%script echo ...`) өшіріңіз.

In [ ]:
%%script echo "Міндетті емес ұяшық: іске қосу үшін осы бірінші жолды өшіріңіз."
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def llm_assistant(question, k=3):
    if re.search(PRIVATE, question.lower()):
        return "Кешіріңіз, адамдардың жеке деректері туралы ақпарат бермеймін."
    hits = index.search(question, k)
    if hits[0][1] < THRESHOLD:
        return "Мектеп құжаттарынан бұл сұраққа жауап таппадым. " + CONTACT
    sources = "\n".join(f"[{n}] {sections[i]['doc']} → {sections[i]['title']}: "
                        f"{sections[i]['text']}" for n, (i, _) in enumerate(hits, 1))
    prompt = ("Сен — мектептің көмекшісісің. Сұраққа тек төмендегі дереккөздерге "
              "сүйеніп, қысқа әрі сыпайы жауап бер. Дереккөздің нөмірін [1] түрінде "
              "көрсет. Жауап болмаса, «білмеймін» деп, мектеп кеңсесіне жібер.\n\n"
              f"{sources}\n\nСҰРАҚ: {question}")
    reply = client.chat.completions.create(
        model="llama3.1", temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return reply.choices[0].message.content

**Міндетті емес ұяшық.** Бұл код қосымша бағдарламаны, интернеттегі қызметті немесе өз файлыңызды қажет етеді (толығырақ кітапта). Іске қосу үшін ұяшықтың бірінші жолын (`%%script echo ...`) өшіріңіз.

In [ ]:
%%script echo "Міндетті емес ұяшық: іске қосу үшін осы бірінші жолды өшіріңіз."
# pip install gradio
import gradio as gr

gr.ChatInterface(fn=lambda message, history: assistant(message),
                 title="«Білім» мектебінің көмекшісі").launch(share=True)